In [1]:
# 1. Data Preparation & SFC Score Calculation.
#This section loads the VariCarta (hg19) dataset, extracts CADD and SFARI annotations, and computes the baseline SFC variant scores.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# --- KAGGLE FILE PATHS ---
varicarta_file = '/kaggle/input/datasets/refatharaislam/genomic-data-asd/dataset2.tsv'
sfari_file = '/kaggle/input/datasets/refatharaislam/genomic-data-asd/SFARI-Gene_genes_05-01-2026release_05-09-2026export.csv'
clinvar_file = '/kaggle/input/datasets/refatharaislam/genomic-data-asd/variant_summary.txt'

print("--- 1. FILTERING CASCADE ---")

# Stage 1: Raw VariCarta
skip_rows = 0
with open(varicarta_file, 'r') as file:
    for i, line in enumerate(file):
        if line.startswith('#CHROM'):
            skip_rows = i
            break
df_raw = pd.read_csv(varicarta_file, sep='\t', skiprows=skip_rows, low_memory=False)
df_raw = df_raw.replace('.', np.nan)

# Restore strict INFO extraction
df_raw['subject_id'] = df_raw['INFO'].str.extract(r'subject_id=([^;]+)')
df_raw['gene_symbol'] = df_raw['INFO'].str.extract(r'gene_symbol=([^;]+)')
df_raw['func'] = df_raw['INFO'].str.extract(r'func=([^;]+)')
df_raw['inheritance'] = df_raw['INFO'].str.extract(r'inheritance=([^;]+)')
df_raw['CADD13_phred'] = pd.to_numeric(df_raw['INFO'].str.extract(r'CADD_phred=([^;]+)')[0], errors='coerce').fillna(10.0)

print(f"Stage 1 (Raw VariCarta): {len(df_raw)} variants, {df_raw['subject_id'].nunique()} subjects")

# Stage 2: Strict Inheritance Filter
def map_inheritance(val):
    v = str(val).lower().strip()
    if v in ['d', 'dn'] or 'de novo' in v: return 'De novo'
    if v in ['i', 'inh', 'm', 'p', 'mat', 'pat'] or 'inherited' in v: return 'Inherited'
    return 'Other'

df_raw['inheritance_clean'] = df_raw['inheritance'].apply(map_inheritance)
df_inh = df_raw[df_raw['inheritance_clean'].isin(['De novo', 'Inherited'])].copy()
df_inh = df_inh.dropna(subset=['gene_symbol'])

print(f"Stage 2 (Inheritance Class Filter): {len(df_inh)} variants, {df_inh['subject_id'].nunique()} subjects")

# Stage 3: SFARI Inner Join
def map_func_vep(v):
    v = str(v).lower()
    if any(x in v for x in ['stopgain', 'frameshift', 'splicing', 'nonsense']): return 3.0
    if any(x in v for x in ['missense', 'nonsynonymous']): return 2.0
    return 1.0
df_inh['func_multiplier'] = df_inh['func'].apply(map_func_vep)

df_sfari_db = pd.read_csv(sfari_file)
df_inh['gene_symbol'] = df_inh['gene_symbol'].astype(str).str.strip().str.upper()
df_sfari_db['gene-symbol'] = df_sfari_db['gene-symbol'].astype(str).str.strip().str.upper()

df_sfari = pd.merge(df_inh, df_sfari_db[['gene-symbol', 'gene-score']], left_on='gene_symbol', right_on='gene-symbol', how='inner')
point_map = {'1': 3.0, 1: 3.0, 1.0: 3.0, '2': 2.0, 2: 2.0, 2.0: 2.0, '3': 1.0, 3: 1.0, 3.0: 1.0}
df_sfari['sfari_points'] = df_sfari['gene-score'].map(point_map).fillna(1.0)

print(f"Stage 3 (SFARI Inner Join): {len(df_sfari)} variants, {df_sfari['subject_id'].nunique()} subjects")

# Stage 4: ClinVar Coordinate Merge
df_clinvar_db = pd.read_csv(clinvar_file, sep='\t', low_memory=False)
def map_cv(v):
    v = str(v).lower()
    if 'pathogenic' in v and 'benign' not in v: return 1
    if 'benign' in v and 'pathogenic' not in v: return 0
    return np.nan
df_clinvar_db['target'] = df_clinvar_db['ClinicalSignificance'].apply(map_cv)
df_clinvar_db = df_clinvar_db.dropna(subset=['target']).rename(columns={'Chromosome': '#CHROM', 'PositionVCF': 'POS', 'ReferenceAlleleVCF': 'REF', 'AlternateAlleleVCF': 'ALT'})

for col in ['#CHROM', 'POS', 'REF', 'ALT']:
    df_clinvar_db[col] = df_clinvar_db[col].astype(str).str.strip()
    df_sfari[col] = df_sfari[col].astype(str).str.strip()

df_val = pd.merge(df_sfari, df_clinvar_db[['#CHROM', 'POS', 'REF', 'ALT', 'target']], on=['#CHROM', 'POS', 'REF', 'ALT'], how='inner')
df_val['SFC_Score'] = df_val['sfari_points'] * df_val['func_multiplier'] * (df_val['CADD13_phred'] / 10.0)
df_val['Score_CADD_Func'] = df_val['func_multiplier'] * (df_val['CADD13_phred'] / 10.0)
df_val['Score_CADD_SFARI'] = df_val['sfari_points'] * (df_val['CADD13_phred'] / 10.0)

print(f"Stage 4 (ClinVar Validation): {len(df_val)} variants, {df_val['subject_id'].nunique()} subjects")

print("\n--- 2. ADDITIVE ABLATION ---")
y_true = df_val['target'].values
for name, col in [('CADD Only', 'CADD13_phred'), 
                  ('CADD x Func', 'Score_CADD_Func'), 
                  ('CADD x SFARI', 'Score_CADD_SFARI'), 
                  ('Full SFC Score', 'SFC_Score')]:
    auc_val = roc_auc_score(y_true, df_val[col])
    ap_val = average_precision_score(y_true, df_val[col])
    print(f"{name:<22}: AUROC = {auc_val:.4f} | AUPRC = {ap_val:.4f}")

print("\n--- 3. PAIRED BOOTSTRAP TEST (SFC vs CADD) ---")
def paired_bootstrap(y, scores_a, scores_b, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n):
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2: continue
        auc_a = roc_auc_score(y[idx], scores_a[idx])
        auc_b = roc_auc_score(y[idx], scores_b[idx])
        deltas.append(auc_a - auc_b)
    deltas = np.array(deltas)
    mean_delta = np.mean(deltas)
    ci_bounds = np.percentile(deltas, [2.5, 97.5])
    p_value = 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())
    return mean_delta, ci_bounds, p_value

sfc_scores = df_val['SFC_Score'].values
cadd_scores = df_val['CADD13_phred'].values 
mean_delta, ci, p = paired_bootstrap(y_true, sfc_scores, cadd_scores)

print(f"Mean ΔAUROC: {mean_delta:.4f}")
print(f"95% CI of ΔAUROC: [{ci[0]:.4f}, {ci[1]:.4f}]")
if p == 0:
    print("p-value: < 0.001")
else:
    print(f"p-value: {p:.4e}")

print("\n--- 4. SUBJECT-LEVEL AGGREGATION DIAGNOSTIC ---")
subj_sfc = df_val.groupby('subject_id')['SFC_Score'].sum().reset_index()
subj_cadd = df_val.groupby('subject_id')['CADD13_phred'].sum().reset_index()
subj_path = df_val[df_val['target'] == 1].groupby('subject_id').size().reset_index(name='path_count')
subj_count = df_val.groupby('subject_id').size().reset_index(name='var_count')

subj_df = subj_sfc.merge(subj_path, on='subject_id', how='left').fillna({'path_count': 0})
subj_df = subj_df.merge(subj_cadd, on='subject_id')
subj_df = subj_df.merge(subj_count, on='subject_id')

rho_sfc, p_sfc = spearmanr(subj_df['SFC_Score'], subj_df['path_count'])
rho_cadd, p_cadd = spearmanr(subj_df['CADD13_phred'], subj_df['path_count'])
print(f"SFC vs Pathogenic Count: rho = {rho_sfc:.4f}, p = {p_sfc:.2e}")
print(f"CADD vs Pathogenic Count: rho = {rho_cadd:.4f}, p = {p_cadd:.2e}")

print("\n--- 5. GENERATING FIGURES ---")
fpr_sfc, tpr_sfc, _ = roc_curve(y_true, sfc_scores)
fpr_cadd, tpr_cadd, _ = roc_curve(y_true, cadd_scores)
plt.figure(figsize=(6,6))
plt.plot(fpr_sfc, tpr_sfc, color='blue', label=f'SFC (AUC={auc(fpr_sfc,tpr_sfc):.4f})')
plt.plot(fpr_cadd, tpr_cadd, color='red', linestyle='dashed', label=f'CADD (AUC={auc(fpr_cadd,tpr_cadd):.4f})')
plt.plot([0,1],[0,1],'k:', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve: Variant Pathogenicity')
plt.legend(loc='lower right'); plt.tight_layout(); plt.savefig('/kaggle/working/ROC_Comparison.png', dpi=300); plt.close()

prec_sfc, rec_sfc, _ = precision_recall_curve(y_true, sfc_scores)
prec_cadd, rec_cadd, _ = precision_recall_curve(y_true, cadd_scores)
plt.figure(figsize=(6,6))
plt.plot(rec_sfc, prec_sfc, color='blue', label=f'SFC (AP={average_precision_score(y_true,sfc_scores):.4f})')
plt.plot(rec_cadd, prec_cadd, color='red', linestyle='dashed', label=f'CADD (AP={average_precision_score(y_true,cadd_scores):.4f})')
plt.axhline(y_true.mean(), linestyle=':', color='gray', label=f'Baseline ({y_true.mean():.4f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve')
plt.legend(loc='lower left'); plt.tight_layout(); plt.savefig('/kaggle/working/PR_Comparison.png', dpi=300); plt.close()
print("Saved ROC_Comparison.png and PR_Comparison.png to /kaggle/working/")

--- 1. FILTERING CASCADE ---
Stage 1 (Raw VariCarta): 527231 variants, 35903 subjects
Stage 2 (Inheritance Class Filter): 487705 variants, 31375 subjects
Stage 3 (SFARI Inner Join): 77525 variants, 15395 subjects
Stage 4 (ClinVar Validation): 5502 variants, 2866 subjects

--- 2. ADDITIVE ABLATION ---
CADD Only             : AUROC = 0.8070 | AUPRC = 0.9074
CADD x Func           : AUROC = 0.8158 | AUPRC = 0.9144
CADD x SFARI          : AUROC = 0.8337 | AUPRC = 0.9226
Full SFC Score        : AUROC = 0.8421 | AUPRC = 0.9291

--- 3. PAIRED BOOTSTRAP TEST (SFC vs CADD) ---
Mean ΔAUROC: 0.0352
95% CI of ΔAUROC: [0.0281, 0.0429]
p-value: < 0.001

--- 4. SUBJECT-LEVEL AGGREGATION DIAGNOSTIC ---
SFC vs Pathogenic Count: rho = 0.5856, p = 1.69e-263
CADD vs Pathogenic Count: rho = 0.5577, p = 5.30e-234

--- 5. GENERATING FIGURES ---
Saved ROC_Comparison.png and PR_Comparison.png to /kaggle/working/


In [2]:
# 2. Missense Benchmark (SFC vs. AlphaMissense vs. CADD)
#This section isolates single nucleotide missense variants and benchmarks SFC against AlphaMissense and CADD using the exact matched coordinates.
import os, numpy as np, pandas as pd
import subprocess
from sklearn.metrics import roc_auc_score, average_precision_score

print("--- PATH A: ALPHAMISSENSE BENCHMARK (DIAGNOSTIC & ADDITIVE TEST) ---")

# --- 1. Isolate SNVs & Normalize ---
df_missense = df_val[
    (df_val['REF'].str.len() == 1) &
    (df_val['ALT'].str.len() == 1)
].copy()

# Normalize coordinates
df_missense['CHROM_clean'] = df_missense['#CHROM'].astype(str).str.replace('chr','',regex=False).str.strip().replace({'23':'X','24':'Y'})
df_missense['REF_clean'] = df_missense['REF'].astype(str).str.upper().str.strip()
df_missense['ALT_clean'] = df_missense['ALT'].astype(str).str.upper().str.strip()

# Safe integer string conversion for POS
df_missense['POS_clean'] = pd.to_numeric(df_missense['POS'], errors='coerce').fillna(0).astype(int).astype(str)
df_missense['merge_key'] = df_missense['CHROM_clean'] + "_" + df_missense['POS_clean'] + "_" + df_missense['REF_clean'] + "_" + df_missense['ALT_clean']

print(f"Total SNVs ready for cross-matching: {len(df_missense)}")

# --- 2. Extract REVEL if present ---
df_missense['REVEL'] = pd.to_numeric(
    df_missense['INFO'].str.extract(r'REVEL=([^;]+)')[0], errors='coerce')

# --- 3. Download AlphaMissense hg19 ---
am_file = '/kaggle/working/AlphaMissense_hg19.tsv.gz'
if not os.path.exists(am_file):
    print("Downloading AlphaMissense hg19...")
    !wget -qO {am_file} "https://zenodo.org/records/8208688/files/AlphaMissense_hg19.tsv.gz?download=1"

# --- 4. The AWK Fast-Pass ---
wanted_file = '/kaggle/working/wanted.txt'
with open(wanted_file, 'w') as f:
    for k in df_missense['merge_key'].unique():
        f.write(k + "\n")

print("Running fast awk filter (this will take ~1-2 minutes)...")
awk_cmd = f"""zcat {am_file} | awk -F'\\t' 'NR==FNR{{w[$1]=1; next}} !/^#/ {{c=$1; sub("^chr","",c); k=c"_"$2"_"toupper($3)"_"toupper($4); if (k in w) print $0}}' {wanted_file} - > /kaggle/working/am_matched.tsv"""
subprocess.run(awk_cmd, shell=True)

# --- 5. Bulletproof Pandas Merge & Diagnostics ---
try:
    df_am_raw = pd.read_csv('/kaggle/working/am_matched.tsv', sep='\t', header=None, dtype=str)
except pd.errors.EmptyDataError:
    df_am_raw = pd.DataFrame()

print(f"AlphaMissense variants successfully extracted: {len(df_am_raw)}")

if len(df_am_raw) == 0:
    print("\nCRITICAL ERROR: No matching variants found.")
else:
    # DIAGNOSTIC DUMP
    print("\n--- AlphaMissense raw column dump ---")
    print(df_am_raw.head(3).to_string())
    print(f"Shape: {df_am_raw.shape}")

    df_am_raw['merge_key'] = df_am_raw[0].astype(str).str.replace('chr','',regex=False) + "_" + \
                             df_am_raw[1].astype(str) + "_" + \
                             df_am_raw[2].astype(str).str.upper() + "_" + \
                             df_am_raw[3].astype(str).str.upper()
    
    # TARGET INDEX 8 FOR SCORES
    df_am_raw['am_pathogenicity'] = pd.to_numeric(df_am_raw[8], errors='coerce')
    
    df_am_clean = df_am_raw[['merge_key', 'am_pathogenicity']].drop_duplicates('merge_key')
    df_missense_clean = df_missense.drop_duplicates('merge_key')

    # Merge on the identical string key
    df_benchmark = pd.merge(df_missense_clean, df_am_clean, on='merge_key', how='inner')
    
    print(f"\nSUCCESS: Final inner merge yielded {len(df_benchmark)} benchmarkable variants!")
    print(f"Pathogenic: {(df_benchmark['target']==1).sum()} | Benign: {(df_benchmark['target']==0).sum()}")

    # --- THE ADDITIVE EXPERIMENT ---
    df_benchmark['SFC_additive'] = df_benchmark['CADD13_phred'] + (df_benchmark['sfari_points'] - 1.0) * 5.0

    # --- 6. Compute AUROC Per Tool ---
    if len(df_benchmark) > 0:
        tools = {
            'CADD Baseline': df_benchmark['CADD13_phred'].values.astype(float),
            'SFC (Multiplicative)':  df_benchmark['SFC_Score'].values.astype(float),
            'SFC (Additive Test)': df_benchmark['SFC_additive'].values.astype(float),
            'AlphaMissense': df_benchmark['am_pathogenicity'].values.astype(float),
        }
        if df_benchmark['REVEL'].notna().sum() > 50:
            tools['REVEL'] = df_benchmark['REVEL'].values.astype(float)

        y = df_benchmark['target'].values.astype(int)
        
        results = []
        for name, scores in tools.items():
            mask = ~np.isnan(scores)
            if mask.sum() < 50 or len(np.unique(y[mask])) < 2: continue
            results.append({
                'Tool': name, 
                'N': int(mask.sum()),
                'AUROC': roc_auc_score(y[mask], scores[mask]),
                'AUPRC': average_precision_score(y[mask], scores[mask]),
            })
        res_df = pd.DataFrame(results)
        print("\n--- HEAD-TO-HEAD BENCHMARK RESULTS ---")
        print(res_df.to_string(index=False))

        # --- 7. Paired bootstrap ---
        def paired_bootstrap(y, s_a, s_b, n=1000, seed=42):
            rng = np.random.default_rng(seed)
            deltas = []
            for _ in range(n):
                idx = rng.choice(len(y), len(y), replace=True)
                if len(np.unique(y[idx])) < 2: continue
                deltas.append(roc_auc_score(y[idx], s_a[idx]) - roc_auc_score(y[idx], s_b[idx]))
            deltas = np.array(deltas)
            return deltas.mean(), np.percentile(deltas,[2.5,97.5]), 2*min((deltas<=0).mean(),(deltas>=0).mean())

        print("\n--- STATISTICAL SIGNIFICANCE: SFC vs AM ---")
        m = ~np.isnan(tools['SFC (Multiplicative)']) & ~np.isnan(tools['AlphaMissense'])
        if m.sum() > 50:
            md, ci, p = paired_bootstrap(y[m], tools['SFC (Multiplicative)'][m], tools['AlphaMissense'][m])
            p_str = "< 0.001" if p == 0 else f"{p:.4f}"
            print(f"Intersection N = {m.sum()}")
            print(f"Mean ΔAUROC = {md:.4f}, 95% CI [{ci[0]:.4f}, {ci[1]:.4f}], p = {p_str}")

        print("\n--- STATISTICAL SIGNIFICANCE: SFC Additive vs CADD ---")
        m2 = ~np.isnan(tools['SFC (Additive Test)']) & ~np.isnan(tools['CADD Baseline'])
        if m2.sum() > 50:
            md2, ci2, p2 = paired_bootstrap(y[m2], tools['SFC (Additive Test)'][m2], tools['CADD Baseline'][m2])
            p_str2 = "< 0.001" if p2 == 0 else f"{p2:.4f}"
            print(f"Intersection N = {m2.sum()}")
            print(f"Mean ΔAUROC = {md2:.4f}, 95% CI [{ci2[0]:.4f}, {ci2[1]:.4f}], p = {p_str2}")

        # --- 8. Save Output ---
        res_df.to_csv('/kaggle/working/path_a_benchmark.csv', index=False)
        print("\nSaved to /kaggle/working/path_a_benchmark.csv")

--- PATH A: ALPHAMISSENSE BENCHMARK (DIAGNOSTIC & ADDITIVE TEST) ---
Total SNVs ready for cross-matching: 4646
Running fast awk filter (this will take ~1-2 minutes)...
AlphaMissense variants successfully extracted: 877

--- AlphaMissense raw column dump ---
      0         1  2  3     4       5                  6       7       8           9
0  chr1   8418972  G  A  hg19  Q9P2R6  ENST00000400908.2  A1208V  0.0916      benign
1  chr1  11168343  C  A  hg19  P42345  ENST00000361445.4  G2510V  0.9993  pathogenic
2  chr1  11189846  A  C  hg19  P42345  ENST00000361445.4  F1888C  0.9906  pathogenic
Shape: (877, 10)

SUCCESS: Final inner merge yielded 836 benchmarkable variants!
Pathogenic: 579 | Benign: 257

--- HEAD-TO-HEAD BENCHMARK RESULTS ---
                Tool   N    AUROC    AUPRC
       CADD Baseline 836 0.723873 0.831998
SFC (Multiplicative) 836 0.680695 0.820779
 SFC (Additive Test) 836 0.701481 0.826961
       AlphaMissense 836 0.831872 0.924223

--- STATISTICAL SIGNIFICANCE: SFC v

In [3]:
# 3. Subject-Level Aggregation Diagnostics
#This section evaluates 8 different patient-level aggregation schemes to determine if SFC scores retain independent biological signal after controlling for total variant count per subject.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from numpy.linalg import lstsq
import warnings
warnings.filterwarnings('ignore')

print("--- PATH B: SUBJECT-LEVEL AGGREGATION DIAGNOSTIC ---")

# --- 1. BUILD SUBJECT-LEVEL AGGREGATION SCHEMES ---
# Ensure grouping runs across the validated ClinVar cohort
grouped = df_val.groupby('subject_id')

subject_data = []

for subject_id, group in grouped:
    scores = group['SFC_Score'].values
    sfari_pts = group['sfari_points'].values
    n_vars = len(scores)
    n_path = (group['target'] == 1).sum()
    
    # 8 Aggregation Schemes
    s_sum = np.sum(scores)
    s_mean = np.mean(scores)
    s_max = np.max(scores)
    s_top5 = np.sum(np.sort(scores)[::-1][:5])
    s_top10 = np.sum(np.sort(scores)[::-1][:10])
    s_logsum = np.log1p(s_sum)
    s_weighted = np.sum(sfari_pts * scores)
    
    subject_data.append({
        'subject_id': subject_id,
        'var_count': n_vars,
        'path_count': n_path,
        'Sum': s_sum,
        'Mean': s_mean,
        'Max': s_max,
        'Top-5 Sum': s_top5,
        'Top-10 Sum': s_top10,
        'Log-Sum': s_logsum,
        'Weighted Sum': s_weighted
    })

df_subj = pd.DataFrame(subject_data)

# Rank-Sum: Cohort-level percentile/rank sum per subject
rank_series = df_val['SFC_Score'].rank(pct=True)
df_val_temp = df_val[['subject_id']].copy()
df_val_temp['rank_pct'] = rank_series
rank_sum_df = df_val_temp.groupby('subject_id')['rank_pct'].sum().reset_index()
df_subj = df_subj.merge(rank_sum_df.rename(columns={'rank_pct': 'Rank-Sum'}), on='subject_id')

# Negative Control: Uniform random subject assignment
np.random.seed(42)
df_subj['Random Control'] = np.random.rand(len(df_subj))

schemes = ['Sum', 'Mean', 'Max', 'Top-5 Sum', 'Top-10 Sum', 'Log-Sum', 'Rank-Sum', 'Weighted Sum', 'Random Control']
print(f"Total subjects evaluated: {len(df_subj)}")

# --- 2. STATISTICAL MACHINERY: PARTIAL SPEARMAN & BOOTSTRAPPING ---
def partial_spearman(x, y, z):
    """Computes partial Spearman correlation of x and y controlling for z."""
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    rz = np.argsort(np.argsort(z)).astype(float)
    
    rz_design = np.column_stack([np.ones_like(rz), rz])
    res_x = rx - rz_design @ lstsq(rz_design, rx, rcond=None)[0]
    res_y = ry - rz_design @ lstsq(rz_design, ry, rcond=None)[0]
    
    return spearmanr(res_x, res_y)

def bootstrap_spearman_ci(x, y, z=None, n=1000, seed=42):
    """Bootstraps 95% CI for bivariate or partial Spearman correlation."""
    rng = np.random.default_rng(seed)
    n_samples = len(x)
    rhos = []
    
    for _ in range(n):
        idx = rng.choice(n_samples, n_samples, replace=True)
        if len(np.unique(x[idx])) < 2 or len(np.unique(y[idx])) < 2:
            continue
        try:
            if z is None:
                r, _ = spearmanr(x[idx], y[idx])
            else:
                if len(np.unique(z[idx])) < 2:
                    continue
                r, _ = partial_spearman(x[idx], y[idx], z[idx])
            rhos.append(r)
        except Exception:
            continue
            
    if len(rhos) == 0:
        return np.nan, np.nan
    return np.percentile(rhos, 2.5), np.percentile(rhos, 97.5)

# --- 3. EVALUATION MATRIX ---
y_outcome = df_subj['path_count'].values
z_confound = df_subj['var_count'].values

results_list = []
print("\nComputing correlations and 95% bootstrap CIs (this may take ~1 minute)...")

for scheme in schemes:
    x_score = df_subj[scheme].values
    
    # Raw correlation
    r_raw, p_raw = spearmanr(x_score, y_outcome)
    ci_raw_low, ci_raw_high = bootstrap_spearman_ci(x_score, y_outcome, z=None)
    
    # Partial correlation controlling for variant count
    r_part, p_part = partial_spearman(x_score, y_outcome, z_confound)
    ci_part_low, ci_part_high = bootstrap_spearman_ci(x_score, y_outcome, z_confound)
    
    results_list.append({
        'Scheme': scheme,
        'Raw_rho': r_raw,
        'Raw_CI': f"[{ci_raw_low:.3f}, {ci_raw_high:.3f}]",
        'Raw_rho2': r_raw**2,
        'Raw_p': p_raw,
        'Partial_rho': r_part,
        'Partial_CI': f"[{ci_part_low:.3f}, {ci_part_high:.3f}]",
        'Partial_rho2': r_part**2,
        'Partial_p': p_part
    })

res_summary = pd.DataFrame(results_list)
res_summary.to_csv('/kaggle/working/path_b_aggregation_diagnostic.csv', index=False)

print("\n--- RESULTS MATRIX ---")
format_cols = ['Scheme', 'Raw_rho', 'Raw_CI', 'Raw_rho2', 'Partial_rho', 'Partial_CI', 'Partial_rho2', 'Partial_p']
print(res_summary[format_cols].to_string(index=False))

# --- 4. PUBLICATION FIGURES ---
print("\n--- GENERATING PUBLICATION FIGURES ---")

# Figure A: Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x_indices = np.arange(len(schemes))
bar_width = 0.35

raw_rhos = res_summary['Raw_rho'].values
part_rhos = res_summary['Partial_rho'].values

rects1 = ax.bar(x_indices - bar_width/2, raw_rhos, bar_width, label='Raw Spearman rho', color='#2b5c8f')
rects2 = ax.bar(x_indices + bar_width/2, part_rhos, bar_width, label='Partial rho (controlling for var_count)', color='#d95f02', hatch='//')

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Spearman Correlation (rho)', fontsize=11)
ax.set_title('Subject-Level Aggregation: Attenuation After Controlling for Variant Count', fontsize=12, fontweight='bold')
ax.set_xticks(x_indices)
ax.set_xticklabels(schemes, rotation=35, ha='right', fontsize=10)
ax.legend(frameon=True)
plt.tight_layout()
plt.savefig('/kaggle/working/Figure_PathB_BarChart.png', dpi=300)
plt.close()

# Figure B: Scatter plot colored by variant count (Sum scheme)
plt.figure(figsize=(7, 5))
sc = plt.scatter(df_subj['Sum'], df_subj['path_count'], c=df_subj['var_count'], cmap='viridis', alpha=0.7, s=25)
cb = plt.colorbar(sc)
cb.set_label('Total Variant Count per Subject', fontsize=10)
plt.xlabel('Subject-Level Summed SFC Score', fontsize=11)
plt.ylabel('ClinVar Pathogenic Variant Count', fontsize=11)
plt.title('Score vs Pathogenicity Colinearity with Cohort Variant Burden', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/Figure_PathB_ScatterConfound.png', dpi=300)
plt.close()

# Figure C: Residual plot
rx = np.argsort(np.argsort(df_subj['Sum'].values)).astype(float)
ry = np.argsort(np.argsort(df_subj['path_count'].values)).astype(float)
rz = np.argsort(np.argsort(df_subj['var_count'].values)).astype(float)
rz_mat = np.column_stack([np.ones_like(rz), rz])
res_x = rx - rz_mat @ lstsq(rz_mat, rx, rcond=None)[0]
res_y = ry - rz_mat @ lstsq(rz_mat, ry, rcond=None)[0]

plt.figure(figsize=(6, 5))
plt.scatter(res_x, res_y, alpha=0.35, color='#4a148c', s=20)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('Residual Ranks (Summed SFC Score | Variant Count)', fontsize=10)
plt.ylabel('Residual Ranks (Pathogenic Count | Variant Count)', fontsize=10)
plt.title('Residualized Orthogonal Association', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/Figure_PathB_Residuals.png', dpi=300)
plt.close()

print("Saved all 3 publication figures to /kaggle/working/")

--- PATH B: SUBJECT-LEVEL AGGREGATION DIAGNOSTIC ---
Total subjects evaluated: 2866

Computing correlations and 95% bootstrap CIs (this may take ~1 minute)...

--- RESULTS MATRIX ---
        Scheme  Raw_rho         Raw_CI  Raw_rho2  Partial_rho      Partial_CI  Partial_rho2     Partial_p
           Sum 0.585600 [0.560, 0.611]  0.342927     0.501299  [0.420, 0.487]      0.251301 3.025855e-182
          Mean 0.400853 [0.371, 0.432]  0.160683     0.432770  [0.356, 0.419]      0.187290 3.665341e-131
           Max 0.407635 [0.376, 0.439]  0.166166     0.428322  [0.352, 0.417]      0.183459 3.112479e-128
     Top-5 Sum 0.582014 [0.556, 0.607]  0.338740     0.495149  [0.416, 0.482]      0.245172 3.599427e-177
    Top-10 Sum 0.585631 [0.560, 0.611]  0.342964     0.501284  [0.420, 0.487]      0.251285 3.118544e-182
       Log-Sum 0.585601 [0.560, 0.611]  0.342928     0.501292  [0.420, 0.487]      0.251294 3.069435e-182
      Rank-Sum 0.618094 [0.590, 0.647]  0.382040     0.538674  [0.455, 0.52